In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    precision_recall_curve, confusion_matrix, classification_report
)
from sklearn.model_selection import StratifiedKFold
import joblib
import json
from pathlib import Path

RANDOM_STATE = 42
SEED = 42
np.random.seed(SEED)

# 3. Prediccion

## 3.0. Glosario

### Conceptos clave en la evaluación de modelos de clasificación

#### `Métricas y elección de AP`
En clasificación binaria se utilizan distintas métricas para evaluar el rendimiento:

- **ROC-AUC**: mide la capacidad global de separar positivos y negativos, sin depender de un umbral específico. Es útil como visión general, aunque en contextos muy desbalanceados puede ser demasiado optimista.  
- **F1-score**: combina precisión y recall en un único valor, pero siempre respecto a un umbral concreto (por ejemplo, 0.5). Refleja qué tan bien equilibra el modelo falsos positivos y falsos negativos en un punto de corte.  
- **AP (Average Precision o área bajo la curva precisión–recall)**: resume el comportamiento de la curva precisión–recall completa, sin fijarse en un único umbral. Es especialmente informativa cuando las clases están desbalanceadas.  

*AP suele ser la métrica prioritaria** para comparar modelos en escenarios de desbalance, ya que refleja de manera más fiel la capacidad del modelo de priorizar correctamente los casos positivos.

#### `Nested CV y Outer CV`
- **Cross-validation** consiste en dividir los datos en varias particiones para entrenar y validar de manera repetida.  
- **Nested cross-validation** añade dos niveles:  
  - El bucle **outer** genera particiones que se reservan para evaluar de forma justa el rendimiento del modelo.  
  - El bucle **inner** utiliza solo la parte de entrenamiento del outer para ajustar hiperparámetros mediante búsqueda.  

De esta forma se separa la fase de ajuste de hiperparámetros de la fase de evaluación, reduciendo el sesgo optimista y ofreciendo una estimación más fiable del rendimiento real.

#### `OOF (out-of-fold predictions)`
En validación cruzada, cada observación pasa alguna vez por un pliegue en el que **no se usa para entrenar**, sino solo para validar.  

Las predicciones obtenidas en esos pliegues se llaman **OOF (out-of-fold)**: son probabilidades que simulan el comportamiento del modelo sobre datos nunca vistos.  

Las OOF permiten calcular métricas de manera más honesta y sirven como base para ajustar parámetros como el umbral de decisión.

#### `Threshold optimizado`
Los modelos de clasificación que devuelven probabilidades necesitan un **umbral** para decidir a partir de qué valor se predice la clase positiva.  
- El valor por defecto suele ser 0.5, pero no siempre es el más adecuado.  
- Usando las predicciones OOF, se puede recorrer una rejilla de umbrales y medir métricas como precisión, recall y F1.  
- El **threshold optimizado** es aquel que maximiza la métrica de interés (por ejemplo, F1).  

De esta manera, en lugar de fijar un valor arbitrario, el modelo se ajusta al equilibrio deseado entre falsos positivos y falsos negativos.

## 3.1. Definición de modelos y *param grids*

In [20]:
logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
rf  = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
hgb = HistGradientBoostingClassifier(random_state=RANDOM_STATE)
svc = SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE)
knn = KNeighborsClassifier()
xgb = XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                    tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1)
lgbm = LGBMClassifier(objective="binary", random_state=RANDOM_STATE, n_jobs=-1)
cat = CatBoostClassifier(loss_function="Logloss", random_state=RANDOM_STATE, verbose=False)
param_grid_logreg = {
    "C": [0.5, 1, 2],  
}
param_grid_rf = {
    "n_estimators": [600],
    "max_depth": [None, 20],
    "min_samples_leaf": [1, 10],
    "max_features": ["sqrt"],
    "class_weight": ["balanced"],
}
param_grid_hgb = {
    "learning_rate": [0.05, 0.1],
    "max_leaf_nodes": [31, 63],
    "min_samples_leaf": [20],
}
param_grid_svc = {
    "C": [1, 10, 100],
    "gamma": ["scale", 0.1],
}
param_grid_knn = {
    "n_neighbors": [15, 31],
    "weights": ["distance"],
    "p": [1, 2],
}
param_grid_xgb = {
    "n_estimators": [800],
    "learning_rate": [0.05, 0.1],
    "max_depth": [4, 6],
    "min_child_weight": [1, 5],
    "subsample": [0.8],
    "colsample_bytree": [0.8],
    "reg_lambda": [1]
}

param_grid_lgbm = {
    "n_estimators": [800],
    "learning_rate": [0.05, 0.1],
    "num_leaves": [31, 63],
    "min_child_samples": [20],
    "colsample_bytree": [0.8],
}
param_grid_cat = {
    "iterations": [800],
    "learning_rate": [0.05, 0.1],
    "depth": [4, 6],
    "l2_leaf_reg": [3],
}

In [21]:
modelos = {
    "LOGREG": (logreg, param_grid_logreg),
    "RF":     (rf,    param_grid_rf),
    "HGB":    (hgb,   param_grid_hgb),
    "SVC":    (svc,   param_grid_svc),
    "KNN":    (knn,   param_grid_knn),
    "XGB":    (xgb,   param_grid_xgb),
    "LGBM":   (lgbm,  param_grid_lgbm),
    "CAT":    (cat,   param_grid_cat),
}

## 3.2. UTilidades

In [22]:
def get_xy(sufix):
    X_train = pd.read_excel(f'../data/processed/X_train_{sufix}.xlsx')
    X_test = pd.read_excel(f'../data/processed/X_test_{sufix}.xlsx')
    y_train = pd.read_excel(f'../data/processed/y_train_{sufix}.xlsx')
    y_test = pd.read_excel(f'../data/processed/y_test_{sufix}.xlsx')
    
    return X_train, X_test, y_train, y_test

In [23]:
def proba_pos(clf, X, pos_label=1):
    classes = clf.classes_
    idx = 1 if pos_label not in classes else int(np.where(classes == pos_label)[0][0])
    return clf.predict_proba(X)[:, idx]

## 3.3. Seleccion de modelo

In [24]:
def model_selection(modelos, X_train, y_train):
    y_train_1d = np.asarray(y_train).ravel()

    cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)  
    cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED+1)
    summary_rows = []
    oof_by_model = {}
    per_fold_by_model = {}

    for name, (clf, grid) in modelos.items():
        print(f"\n===== Modelo: {name} =====")
        y_oof = np.full(len(y_train_1d), np.nan, dtype=float)
        folds_scores = []

        for fold, (tr_idx, va_idx) in enumerate(cv_outer.split(X_train, y_train_1d), start=1):
            print(f"Fold {fold} de {cv_outer.n_splits}")
            X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
            y_tr, y_va = y_train_1d[tr_idx], y_train_1d[va_idx]  # 1-D arrays

            gs = GridSearchCV(
                estimator=clf,
                param_grid=grid,               # only estimator params here (no 'clf__')
                cv=cv_inner,
                scoring="average_precision",
                n_jobs=2,
                refit=True,
                verbose=0
            )
            gs.fit(X_tr, y_tr)

            y_va_proba = proba_pos(gs.best_estimator_, X_va, pos_label=1)
            ap   = average_precision_score(y_va, y_va_proba)
            roc  = roc_auc_score(y_va, y_va_proba)
            f1_05 = f1_score(y_va, (y_va_proba >= 0.5).astype(int))

            folds_scores.append({"fold": fold, "AP": ap, "ROC-AUC": roc, "F1@0.5": f1_05})
            y_oof[va_idx] = y_va_proba

        fold_df = pd.DataFrame(folds_scores)
        per_fold_by_model[name] = fold_df.copy()
        summary_rows.append({
            "model": name,
            "AP_mean":    fold_df["AP"].mean(),
            "AP_std":     fold_df["AP"].std(),
            "ROC_mean":   fold_df["ROC-AUC"].mean(),
            "ROC_std":    fold_df["ROC-AUC"].std(),
            "F1@0.5_mean": fold_df["F1@0.5"].mean(),
            "F1@0.5_std":  fold_df["F1@0.5"].std(),
        })
        oof_by_model[name] = y_oof

    summary_df = (pd.DataFrame(summary_rows)
                    .sort_values("AP_mean", ascending=False)
                    .reset_index(drop=True)
                    .round(4))
    best_model_name = summary_df.loc[0, "model"]
    print(f"\nMejor modelo por AP: {best_model_name}")

    return summary_df, oof_by_model, per_fold_by_model, best_model_name


## 3.4. Threshold optimizada

In [25]:
def get_best_threshold(y_train, best_model_name, oof_by_model):
    y_oof_best = oof_by_model[best_model_name]
    if np.isnan(y_oof_best).any():
        raise RuntimeError("OOF del mejor modelo contiene NaNs.")
    prec, rec, thr = precision_recall_curve(y_train, y_oof_best)
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    thr_opt = float(thr[np.argmax(f1s)])
    print(f"Umbral óptimo (max F1 sobre OOF): {thr_opt:.4f}")
    return thr_opt


### 3.5. Busqueda del mejor modelo

In [26]:
def evaluate_best_model(modelos, X_train, y_train, best_model_name):
    # ensure y is 1-D
    y_train_1d = np.asarray(y_train).ravel()

    clf_best, grid_best = modelos[best_model_name]
    final_gs = GridSearchCV(
        estimator=clf_best,
        param_grid=grid_best,  # no 'clf__' keys if estimator isn't a Pipeline
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        scoring="average_precision",
        n_jobs=-1,
        refit=True,
        verbose=0
    )
    final_gs.fit(X_train, y_train_1d)
    final_model = final_gs.best_estimator_
    print("Mejores hiperparámetros (final):", final_gs.best_params_)
    return final_model


## 3.6. Evaluacion en test

In [27]:
def evaluate_on_test(X_tst, y_tst, threshold, model):
    y_proba = proba_pos(model, X_tst, pos_label=1)
    ap = average_precision_score(y_tst, y_proba)
    roc = roc_auc_score(y_tst, y_proba)
    f1_05 = f1_score(y_tst, (y_proba >= 0.5).astype(int))
    f1_opt = f1_score(y_tst, (y_proba >= threshold).astype(int))

    print("\n=== EVALUACIÓN EN TEST ===")
    print(f"TEST AP: {ap:.4f}")
    print(f"TEST ROC-AUC: {roc:.4f}")
    print(f"TEST F1@0.5: {f1_05:.4f}")
    print(f"TEST F1@thr_opt ({threshold:.4f}): {f1_opt:.4f}")

    y_pred_opt = (y_proba >= threshold).astype(int)
    print("\nMatriz de confusión (umbral óptimo):")
    print(confusion_matrix(y_tst, y_pred_opt))
    print("\nClassification report (umbral óptimo):")
    print(classification_report(y_tst, y_pred_opt, digits=4))

    return ap, roc, f1_opt

## 3.7. Guardado del modelo

In [28]:
def save_model(path, model, row, X_train, thr_opt):
    # 4) Save artifacts
    artifacts_dir = Path(path)
    artifacts_dir.mkdir(parents=True, exist_ok=True)

    feat_cols = list(getattr(X_train, "columns", [])) or None
    (artifacts_dir / "feature_columns.json").write_text(json.dumps(feat_cols, indent=2))

    metadata = row
    if hasattr(model, "best_params_"):
        metadata["best_params"] = getattr(model, "best_params_")

    metadata["threshold"] = thr_opt
    joblib.dump(model, artifacts_dir / "model.joblib")
    
    (artifacts_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))

    print(f"Artifacts saved to: {artifacts_dir.resolve()}")
    
    return artifacts_dir

## 3.8. Pipeline

In [29]:
rows = []
kinds = ['cc', 'cs', 'sc', 'ss']
for kind in kinds:
    X_train, X_test, y_train, y_test = get_xy(kind)
    summary_df, oof_by_model, per_fold_by_model, best_model_name = model_selection(modelos, X_train, y_train)
    thr_opt = get_best_threshold(y_train, best_model_name, oof_by_model)
    final_model = evaluate_best_model(modelos, X_train, y_train, best_model_name)
    ap, auc, f1 = evaluate_on_test(X_test, y_test, thr_opt, final_model)

    best_cv = summary_df.loc[summary_df["model"] == best_model_name].iloc[0].to_dict()

    row = {"kind": kind, "best_model": best_model_name, "threshold": thr_opt, "AP_test": ap, "AUC_test": auc, "F1_test": f1}
    row.update({
                "AP_cv_mean":     best_cv.get("AP_mean", np.nan),
                "AP_cv_std":      best_cv.get("AP_std", np.nan),
                "ROC_cv_mean":    best_cv.get("ROC_mean", np.nan),
                "ROC_cv_std":     best_cv.get("ROC_std", np.nan),
                "F1@0.5_cv_mean": best_cv.get("F1@0.5_mean", np.nan),
                "F1@0.5_cv_std":  best_cv.get("F1@0.5_std", np.nan),
            })
    rows.append(row)

    save_model(f"../models/{kind}", final_model, row, X_train, thr_opt)
    print(f"---> Resultados para {kind}: AP={ap:.4f}, AUC={auc:.4f}, F1={f1:.4f}")
    
df_models = pd.DataFrame(rows).round(4)


===== Modelo: LOGREG =====
Fold 1 de 5
Fold 2 de 5
Fold 3 de 5
Fold 4 de 5
Fold 5 de 5

===== Modelo: RF =====
Fold 1 de 5
Fold 2 de 5
Fold 3 de 5
Fold 4 de 5
Fold 5 de 5

===== Modelo: HGB =====
Fold 1 de 5
Fold 2 de 5
Fold 3 de 5
Fold 4 de 5
Fold 5 de 5

===== Modelo: SVC =====
Fold 1 de 5
Fold 2 de 5
Fold 3 de 5
Fold 4 de 5
Fold 5 de 5

===== Modelo: KNN =====
Fold 1 de 5
Fold 2 de 5
Fold 3 de 5
Fold 4 de 5
Fold 5 de 5

===== Modelo: XGB =====
Fold 1 de 5
Fold 2 de 5
Fold 3 de 5
Fold 4 de 5
Fold 5 de 5

===== Modelo: LGBM =====
Fold 1 de 5
[LightGBM] [Info] Number of positive: 3311, number of negative: 3311
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001267 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 811
[LightGBM] [Info] Number of data points in the train set: 6622, number of used features: 26
[LightGBM] [Info] [binary:Boos

In [30]:
df_models = df_models.sort_values("AP_test", ascending=False, inplace=False)
df_models

,kind,best_model,threshold,AP_test,AUC_test,F1_test,AP_cv_mean,AP_cv_std,ROC_cv_mean,ROC_cv_std,F1@0.5_cv_mean,F1@0.5_cv_std
3,ss,RF,0.5138,0.6560,0.8459,0.6340,0.6628,0.0133,0.8468,0.0025,0.6363,0.0164
2,sc,RF,0.5022,0.6534,0.8456,0.6359,0.6622,0.0132,0.8460,0.0024,0.6367,0.0167
0,cc,CAT,0.4721,0.6206,0.8172,0.6114,0.9203,0.0052,0.9230,0.0040,0.8486,0.0056
1,cs,CAT,0.4480,0.6148,0.8197,0.6072,0.9189,0.0034,0.9218,0.0033,0.8455,0.0069


## 3.9. Final Model

El modelo final es uno que obtiene los datos sin SMOTE y sin seleccion de caracteristicas. Y su enfoque es el de un Random Forest CLasifier. A continuacion, aharemos una busqueda de hiperaprametros mas exaustiva sobre este tipo.

In [33]:
modelo_final  = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

param_grid_mf = {
    "n_estimators": [300, 600],
    "max_depth": [None, 20],
    "min_samples_leaf": [1, 4],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced_subsample"],
}
modelo_final = {
    "RF":     (modelo_final, param_grid_mf),
}
kind_final = ['ss']


In [34]:
rows = []
for kind in kind_final:
    X_train, X_test, y_train, y_test = get_xy(kind)
    summary_df, oof_by_model, per_fold_by_model, best_model_name = model_selection(modelo_final, X_train, y_train)
    thr_opt = get_best_threshold(y_train, best_model_name, oof_by_model)
    final_model = evaluate_best_model(modelo_final, X_train, y_train, best_model_name)
    ap, auc, f1 = evaluate_on_test(X_test, y_test, thr_opt, final_model)

    best_cv = summary_df.loc[summary_df["model"] == best_model_name].iloc[0].to_dict()

    row = {"kind": kind, "best_model": best_model_name, "threshold": thr_opt, "AP_test": ap, "AUC_test": auc, "F1_test": f1}
    row.update({
                "AP_cv_mean":     best_cv.get("AP_mean", np.nan),
                "AP_cv_std":      best_cv.get("AP_std", np.nan),
                "ROC_cv_mean":    best_cv.get("ROC_mean", np.nan),
                "ROC_cv_std":     best_cv.get("ROC_std", np.nan),
                "F1@0.5_cv_mean": best_cv.get("F1@0.5_mean", np.nan),
                "F1@0.5_cv_std":  best_cv.get("F1@0.5_std", np.nan),
            })
    rows.append(row)

    save_model("../models/best", final_model, row, X_train, thr_opt)
    print(f"---> Resultados para modelo final: AP={ap:.4f}, AUC={auc:.4f}, F1={f1:.4f}")
    
final_df = pd.DataFrame(rows)


===== Modelo: RF =====
Fold 1 de 5
Fold 2 de 5
Fold 3 de 5
Fold 4 de 5
Fold 5 de 5

Mejor modelo por AP: RF
Umbral óptimo (max F1 sobre OOF): 0.3141
Mejores hiperparámetros (final): {'class_weight': None, 'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 4, 'n_estimators': 600}

=== EVALUACIÓN EN TEST ===
TEST AP: 0.6519
TEST ROC-AUC: 0.8431
TEST F1@0.5: 0.5808
TEST F1@thr_opt (0.3141): 0.6290

Matriz de confusión (umbral óptimo):
[[790 245]
 [ 90 284]]

Classification report (umbral óptimo):
              precision    recall  f1-score   support

           0     0.8977    0.7633    0.8251      1035
           1     0.5369    0.7594    0.6290       374

    accuracy                         0.7622      1409
   macro avg     0.7173    0.7613    0.7270      1409
weighted avg     0.8019    0.7622    0.7730      1409

Artifacts saved to: C:\Users\pafeg\Documents\projects\CAPGEMINI\PFG-TestDataScience-1\models\best
---> Resultados para modelo final: AP=0.6519, AUC=0.8431, F1=0.

In [36]:
final_df.round(4)

,kind,best_model,threshold,AP_test,AUC_test,F1_test,AP_cv_mean,AP_cv_std,ROC_cv_mean,ROC_cv_std,F1@0.5_cv_mean,F1@0.5_cv_std
0,ss,RF,0.3141,0.6519,0.8431,0.629,0.6579,0.014,0.8424,0.0044,0.5811,0.0338


Sin embargo, los resultados no mejoran.

## 3.10. Prediccion

In [16]:
from pprint import pprint
pprint([X_test.iloc[0,:].to_dict()])

[{'b_dependents': 1.0,
  'b_devprot': 1.0,
  'b_eticket': 1.0,
  'b_gender': 1.0,
  'b_mphone': 1.0,
  'b_onlineback': 1.0,
  'b_onlinesec': 1.0,
  'b_partner': 1.0,
  'b_phone': 1.0,
  'b_senior': 0.0,
  'b_streamingmovies': 1.0,
  'b_streamingtv': 1.0,
  'b_techsup': 1.0,
  'c_contract_1y': 0.0,
  'c_contract_2y': 1.0,
  'c_contract_monthly': 0.0,
  'c_internet_dsl': 0.0,
  'c_internet_fiber': 1.0,
  'c_internet_no': 0.0,
  'c_paymethod_btransfer': 0.0,
  'c_paymethod_ccard': 1.0,
  'c_paymethod_echeck': 0.0,
  'c_paymethod_mcheck': 0.0,
  'f_monthly': 0.8014722797331493,
  'f_total': 2.060467250243966,
  'i_tenure': 0.9347826086956522}]


In [ ]:
def predict_batch(model_path, examples):
    model = joblib.load(model_path)
    df = pd.DataFrame(examples)
    predictions = model.predict_proba(df)[:, 1].round(6).tolist()
    return predictions

predict_batch('../models/best/model.joblib', [X_test.iloc[0,:].to_dict()])

[0.039687]